# Exploratory Data Analysis: CIC-IDS-2017
## Dataset Overview and Statistical Analysis

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
warnings.filterwarnings('ignore')

print("✅ Bibliothèques chargées avec succès.")

✅ Bibliothèques chargées avec succès.


## 2. Load Dataset

In [3]:
# 2. Chargement des données (Échantillon de 500 000 lignes)
file_path = '../data/cicids2017.csv' 
target_col = 'Attack Type' # Le nom de la colonne qui contient les attaques

print(f"📂 Chargement depuis : {file_path}")

try:
    # Lecture d'un échantillon pour ne pas saturer la RAM
    df = pd.read_csv(file_path, nrows=500000)
    
    # NETTOYAGE CRITIQUE : Enlever les espaces avant/après les noms de colonnes
    df.columns = df.columns.str.strip()
    
    print(f"✅ Chargement réussi ! Taille de l'échantillon : {df.shape[0]} lignes et {df.shape[1]} colonnes.")
    display(df.head(3))
    
except Exception as e:
    print(f"❌ Erreur lors du chargement : {e}")

📂 Chargement depuis : ../data/cicids2017.csv
✅ Chargement réussi ! Taille de l'échantillon : 500000 lignes et 53 colonnes.


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.97561,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.97561,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.00000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic


## 3. Le Bilan de Santé (NaN et Infs)

In [ ]:
# 3. Bilan de Santé des Données (Check for Missing/Infinite Values)
print("🔍 Recherche des valeurs problématiques pour le Machine Learning...\n")

# Recherche des valeurs vides (NaN)
nan_counts = df.isna().sum()
cols_with_nans = nan_counts[nan_counts > 0]
print("👉 Colonnes avec des valeurs vides (NaN) :")
if cols_with_nans.empty:
    print("   ✅ Aucune.")
else:
    print(cols_with_nans)

# Recherche des valeurs Infinies
numeric_cols = df.select_dtypes(include=[np.number]).columns
inf_counts = df[numeric_cols].isin([np.inf, -np.inf]).sum()
cols_with_infs = inf_counts[inf_counts > 0]

print("\n👉 Colonnes avec des valeurs infinies (Inf) :")
if cols_with_infs.empty:
    print("   ✅ Aucune.")
else:
    print(cols_with_infs)

: 

## 4. Le Déséquilibre des Classes

In [ ]:
# 4. Distribution de la variable cible (Attaques vs Trafic Normal)
plt.figure(figsize=(12, 8))

# Graphique horizontal (barh)
ax = df[target_col].value_counts().plot(kind='barh', color='steelblue', edgecolor='black')
plt.title('Distribution des types de trafic (Preuve du déséquilibre)', fontsize=14, fontweight='bold')
plt.xlabel('Nombre de flux réseaux')
plt.ylabel('Type de trafic')

plt.gca().invert_yaxis() # Inverser pour avoir la plus grande barre en haut

# Ajout des valeurs au bout des barres
for i, v in enumerate(df[target_col].value_counts()):
    ax.text(v + 3, i + 0.1, str(v), color='black', fontweight='bold')

plt.tight_layout()
plt.show()

# Affichage des pourcentages
print("Répartition en pourcentage :")
print(df[target_col].value_counts(normalize=True) * 100)

## 5. Corrélation Cybersécurité

In [ ]:
# 5. Analyse de Corrélation avec les Attaques
print("📊 Calcul des corrélations (Quelles features détectent les attaques ?) ...\n")

# On copie uniquement les colonnes numériques
numeric_df = df.select_dtypes(include=[np.number]).copy()

# On crée une cible binaire temporaire (0 = BENIGN, 1 = Attaque)
numeric_df['Is_Attack'] = df[target_col].apply(lambda x: 0 if x == 'Normal Traffic' else 1)

# Calcul de la matrice
correlation_matrix = numeric_df.corr()

# On extrait les corrélations avec notre cible 'Is_Attack' (en valeur absolue)
target_corr = correlation_matrix['Is_Attack'].abs().sort_values(ascending=False)

print("🏆 Top 10 des caractéristiques les plus liées aux attaques :")
print(target_corr.head(11)[1:]) # [1:] pour exclure la corrélation de 'Is_Attack' avec elle-même

del numeric_df # Libération de la RAM

## 6. Les Colonnes Mortes

In [ ]:
# 6. Recherche des colonnes inutiles (Zero Variance)
print("🗑️ Recherche des colonnes avec une seule valeur constante :\n")

constant_cols = [col for col in df.columns if df[col].nunique() == 1]

if len(constant_cols) > 0:
    print(f"⚠️ {len(constant_cols)} colonnes constantes trouvées (à supprimer) :")
    for col in constant_cols:
        valeur = df[col].iloc[0]
        print(f"  - {col} (Valeur unique : {valeur})")
else:
    print("✅ Aucune colonne constante trouvée.")